
# 練習問題: ループ (while / do while)

## 目標

回数が事前に決まっていない繰り返しを, 条件で続ける `while` (Fortran は `do while`) で書けるようになる.

## 課題

`loop.cpp` (または `loop.f90`) は, 与えられた `N` に対して **2 を何回かけたら `N` を超えるか** (`2^k > N` となる最小の `k`) を求める.
繰り返しの本体が空なので, 現状では `k = 0`, `p = 1` のまま.

`TODO` の箇所に **`p` が `N` を超えるまで「`p` を 2 倍し, `k` を 1 増やす」を繰り返す** ループを書け.

- C++: `while (p <= N) { p = p * 2; k++; }`
- Fortran: `do while (p <= N)` … `p = p * 2` … `k = k + 1` … `end do`

## コンパイルと実行

```
# C++
nvc++ -fast loop.cpp -o loop.exe

# Fortran
nvfortran -fast loop.f90 -o loop.exe
```

```
./loop.exe 1000
./loop.exe 1000000
```

## 期待される結果

`2^10 = 1024` が 1000 を超える最初の 2 のべきなので

```
2^10 = 1024 is the first power of 2 greater than 1000
```

ループを書く前は `2^0 = 1` のままになる. 引数 `N` を変えて結果が変わることを確かめよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ loop.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char ** argv) {
  long N = (argc > 1 ? atol(argv[1]) : 1000);
  /* 2 を何回かけたら N を超えるか (2^k > N となる最小の k) を求める */
  long p = 1;   /* p = 2^k */
  int k = 0;
  // TODO: p が N を超えるまで「p を 2 倍し k を 1 増やす」を繰り返す while ループを書け.
  printf("2^%d = %ld is the first power of 2 greater than %ld\n", k, p, N);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore loop.cpp -o loop_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./loop_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ loop.f90
program loop
  character(len=32) :: arg
  integer(8) :: N, p
  integer :: k
  N = 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N
  end if
  ! 2 を何回かけたら N を超えるか (2^k > N となる最小の k) を求める
  p = 1            ! p = 2^k
  k = 0
  ! TODO: p が N を超えるまで「p を 2 倍し k を 1 増やす」を繰り返す do while ループを書け.
  print "(a,i0,a,i0,a,i0)", "2^", k, " = ", p, " is the first power of 2 greater than ", N
end program loop

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore loop.f90 -o loop_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./loop_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:loop.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:loop.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:loop.cpp}